# AI Project 5

In this project you will complete the provided Python code, using PyTorch, to perform image classification on the Fashion-MNIST using various CNN architectures.

## **Important!**
Make sure you change your runtime type to GPU! This can be found in the top menu following "Runtime->Change runtime type" then select from the dropdown GPU.


## Imports, Data, and Hyperparameters

### Imports

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

import numpy
import matplotlib.pyplot as plt

import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(torch.__version__)
print("Using device:", device)

### Data

In [ ]:
# Download Fashion-MNIST. ToTensor() converts images to float tensors in [0, 1]
# and reshapes from (H, W) to (1, H, W), which is what PyTorch's Conv2d expects.
transform = transforms.ToTensor()

train_dataset = datasets.FashionMNIST(root="./data", train=True, download=True, transform=transform)
test_dataset = datasets.FashionMNIST(root="./data", train=False, download=True, transform=transform)

# get info about data
train_size = len(train_dataset)
test_size = len(test_dataset)
data_shape = train_dataset[0][0].shape  # (1, 28, 28)
num_classes = len(train_dataset.classes)

print(
    "There are ",
    train_size,
    " training examples and ",
    test_size,
    " testing examples with ",
    num_classes,
    " classes",
)

### Global Hyperparameters

In [ ]:
LEARNING_RATE = 0.0001
BATCH_SIZE = 32
EPOCHS = 25

## **Part 1 - The Effect of Filters**

In this section you will be comparing the relative performance and training times for two different CNN architectures.

Both architectures will have the following layers in order:

``` Python
Conv2d <- input layer provided
Conv2d
MaxPool2d((2,2))
Conv2d
Conv2d
MaxPool2d((2,2))
Flatten
Linear(1024 nodes)
Linear(num_classes) <- output layer provided
```

The input layers and output layer will be provided.

In ```build_model_1A()``` the first two Conv2d layers should have 16 filters (``out_channels=16``), and the second two Conv2d layers should have 32 filters.

In ```build_model_1B()``` the first two Conv2d layers should have 64 filters, and the second two Conv2d layers should have 128 filters.

All hidden layers, except the output layer, should be followed by a ```nn.ReLU()``` activation. Note that in PyTorch, ReLU is its own separate module, rather than being an argument on ``nn.Conv2d`` or ``nn.Linear``. All ```nn.Conv2d``` layers should have ```padding=1``` and ```kernel_size=3``` passed as parameters.

The first ``Conv2d`` needs ``in_channels=1`` because the input is grayscale. For every subsequent ``Conv2d``, the ``in_channels`` must match the ``out_channels`` of the previous ``Conv2d``.

Both max pooling layers should have a pool size of (2, 2) (``kernel_size=2``).

**Note on the output layer:** In PyTorch, ``nn.CrossEntropyLoss`` (which we'll use below) applies the softmax internally. So the final ``nn.Linear`` outputs raw scores (logits) - no explicit softmax activation is needed.

**Documentation for each layer type:**

*Conv2d*: https://docs.pytorch.org/docs/stable/generated/torch.nn.Conv2d.html

*MaxPool2d*: https://docs.pytorch.org/docs/stable/generated/torch.nn.MaxPool2d.html

*Flatten*: https://docs.pytorch.org/docs/stable/generated/torch.nn.modules.flatten.Flatten.html

*Linear*: https://docs.pytorch.org/docs/stable/generated/torch.nn.Linear.html

**Documentation for Sequential model**: https://docs.pytorch.org/docs/stable/generated/torch.nn.Sequential.html

### ***TODO*** Model 1A

In [ ]:
def build_model_1A():
  return nn.Sequential(
      # Input layer (provided)
      nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, padding=1),
      nn.ReLU(),
      # !!! Your layers here !!!
      # Output layer (provided). After the final MaxPool2d, feature maps are 7x7 with
      # 32 channels, so the flattened size going into the Linear(1024) is 7*7*32.
      # The line below is the very last layer: maps 1024 features -> num_classes logits.
      nn.Linear(1024, num_classes),
  )

### ***TODO*** Model 1B

In [ ]:
def build_model_1B():
  return nn.Sequential(
      # Input layer (provided)
      nn.Conv2d(in_channels=1, out_channels=64, kernel_size=3, padding=1),
      nn.ReLU(),
      # !!! Your layers here !!!
      # Output layer (provided). After the final MaxPool2d, feature maps are 7x7 with
      # 128 channels, so the flattened size going into the Linear(1024) is 7*7*128.
      nn.Linear(1024, num_classes),
  )

### Model Compilation and Summaries

Compiles and prints summaries for the two architectures. Do not change.


In [ ]:
# get networks
model_1a = build_model_1A().to(device)
model_1b = build_model_1B().to(device)

# print summaries
print("\nModel 1A Architecture\n")
print(model_1a)
print("Total parameters:", sum(p.numel() for p in model_1a.parameters()))

print("\nModel 1B Architecture\n")
print(model_1b)
print("Total parameters:", sum(p.numel() for p in model_1b.parameters()))

### Testing Helper Method

Method to help with testing. Do not change


In [ ]:
train_loader_cached = None
test_loader_cached = None

def get_loaders():
  global train_loader_cached, test_loader_cached
  # (Re)build each time so batch size changes take effect
  train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
  test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
  return train_loader, test_loader

def perf_and_test(model, model_name):
  model = model.to(device)
  optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
  criterion = nn.CrossEntropyLoss()
  train_loader, test_loader = get_loaders()

  history = {"accuracy": [], "loss": []}
  start = time.time()

  for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    running_correct = 0
    running_total = 0
    for x, y in train_loader:
      x, y = x.to(device), y.to(device)
      optimizer.zero_grad()
      logits = model(x)
      loss = criterion(logits, y)
      loss.backward()
      optimizer.step()
      running_loss += loss.item() * x.size(0)
      running_correct += (logits.argmax(dim=1) == y).sum().item()
      running_total += x.size(0)
    epoch_loss = running_loss / running_total
    epoch_acc = running_correct / running_total
    history["loss"].append(epoch_loss)
    history["accuracy"].append(epoch_acc)
    print(f"Epoch {epoch+1}/{EPOCHS} - loss: {epoch_loss:.4f} - accuracy: {epoch_acc:.4f}")

  print("\n" + model_name + " testing information\n")
  print("\nTraining took ", round((time.time() - start), 2), "seconds\n")

  print("\nTesting statistics\n")
  model.eval()
  test_loss = 0.0
  test_correct = 0
  test_total = 0
  with torch.no_grad():
    for x, y in test_loader:
      x, y = x.to(device), y.to(device)
      logits = model(x)
      loss = criterion(logits, y)
      test_loss += loss.item() * x.size(0)
      test_correct += (logits.argmax(dim=1) == y).sum().item()
      test_total += x.size(0)
  test_loss /= test_total
  test_acc = test_correct / test_total
  print(f"test loss: {test_loss:.4f} - test accuracy: {test_acc:.4f}")

  fig, ax = plt.subplots()
  ax.plot(history["accuracy"], color="red")
  ax.set_xlabel("epoch", fontsize=14)
  ax.set_ylabel("accuracy", color="red", fontsize=14)
  ax2 = ax.twinx()
  ax2.plot(history["loss"], color="blue")
  ax2.set_ylabel("loss", color="blue", fontsize=14)
  plt.title("Training Acc. and Loss: " + model_name)
  plt.show()

### ***TODO*** Global Hyperparameters

Start with a number of epochs that seems too high Ex:(between 30 and 80). After training, examine the loss plot to identify a point of diminishing returns in training, and then set the number of training epochs equal to that. Note any changes in testing accuracy.

In [ ]:
EPOCHS = 25

### Testing models

The following code cell will run both models and provided statistics for the training time, testing accuracy, and testing loss of the models. It also generates graphs for training accuracy and loss over time. These graphs can be used in your report.


In [ ]:
### Model building code included to allow for fresh reruns ###

# get fresh networks
model_1a = build_model_1A()
model_1b = build_model_1B()

# test
perf_and_test(model_1a, "Model 1A")
perf_and_test(model_1b, "Model 1B")

## **Part 2 - Dropout Layers**

In part 2, we will be examining the effect of dropout layers on the performance of the model. Copy your code from the ```build_model_1B()``` method into the ```build_model_2()``` method, and then add dropout layers after each max pooling layer. Model 2 architecture should be:

```
Conv2d
Conv2d
MaxPool2d((2,2))
Dropout(0.5) <- Add this
Conv2d
Conv2d
MaxPool2d((2,2))
Dropout(0.5) <- Add this
Flatten
Linear(1024 nodes)
Linear(num_classes)
```

Note that the we are passing ```0.5``` as an argument to the dropout layers. This means that 50% of the values flowing through will be zeroed during training.

*Documentation on dropout layers can be found here*: https://docs.pytorch.org/docs/stable/generated/torch.nn.Dropout.html

### ***TODO*** Model 2

In [ ]:
def build_model_2():
  # Copy your code from build_model_1B here and add dropout layers as specified
  pass

### Model Compilation and Summaries


In [ ]:
# get network
model_2 = build_model_2().to(device)

# print summary
print("\nModel 2 Architecture\n")
print(model_2)
print("Total parameters:", sum(p.numel() for p in model_2.parameters()))

### **TODO** Global Hyperparameters

Dropout layers, while capable of making the model more robust, also require more training. You should again increase the number of epochs to find a point of diminishing returns.

In [ ]:
EPOCHS = 25

### Testing Model 2

When looking at your results, pay attention to the difference between the training and test accuracy of Model 1B to the difference of the two accuracies in Model 2.

In [ ]:
# get fresh network
model_2 = build_model_2()

perf_and_test(model_2, "Model 2")

## **Part 3 - Batch Normalization**

In part 3, we will be examining the effect of batch normalization layers on the performance of the model. Copy your code from the ```build_model_1B()``` method into the ```build_model_3()``` method, and then add batch normalization layers after each Conv2d layer (after the Conv2d, before its ReLU). The Model 3 architecture should be:

```
Conv2d
BatchNorm2d
ReLU
Conv2d
BatchNorm2d
ReLU
MaxPool2d((2,2))
Conv2d
BatchNorm2d
ReLU
Conv2d
BatchNorm2d
ReLU
MaxPool2d((2,2))
Flatten
Linear(1024 nodes)
Linear(num_classes)
```

``nn.BatchNorm2d`` requires the number of feature channels as its only argument - this must match the ``out_channels`` of the preceding ``Conv2d``.

*Documentation can be found here*: https://docs.pytorch.org/docs/stable/generated/torch.nn.BatchNorm2d.html

### ***TODO*** Model 3

In [ ]:
def build_model_3():
  # Copy your code from build_model_1B here and add BatchNorm2d layers as specified
  pass

### Model Compilation and Summary


In [ ]:
# get network
model_3 = build_model_3().to(device)

# print summary
print("\nModel 3 Architecture\n")
print(model_3)
print("Total parameters:", sum(p.numel() for p in model_3.parameters()))

### ***TODO*** Global Hyperparameters

 Batch normalization has been shown to improve convergence in neural networks, and so the number of training epochs required may be lower than in the other architectures. Again refer to the initial loss plot to find the point of diminishing returns and update the ```EPOCHS``` variable.

In [ ]:
EPOCHS = 25

### Testing Model 3


In [ ]:
# get fresh network
model_3 = build_model_3()

perf_and_test(model_3, "Model 3")

## **Part 4 - Layer count**

In the final part of this project, you will be looking out how the number of layers affect performance and training time.

In ```build_model_4```, you need to implement the following architecture:

```python
# block 1 - 64 filters per Conv2d
Conv2d
BatchNorm2d
ReLU
Conv2d
BatchNorm2d
ReLU
Conv2d
BatchNorm2d
ReLU
MaxPool2d
Dropout(0.5)
# block 2 - 128 filters per Conv2d
Conv2d
BatchNorm2d
ReLU
Conv2d
BatchNorm2d
ReLU
Conv2d
BatchNorm2d
ReLU
MaxPool2d
Dropout(0.5)
# block 3 - 256 filters per Conv2d
Conv2d
BatchNorm2d
ReLU
Conv2d
BatchNorm2d
ReLU
Conv2d
BatchNorm2d
ReLU
MaxPool2d
Dropout(0.5)
# Dense layers
Flatten
Linear(1024)
Linear(512)
Linear(num_classes)
```

Notice each block consists of 3 Conv2d layers, with 3 BatchNorm2d layers, a MaxPool2d and Dropout layer. The first block should have 64 filters for each Conv2d layer, the second should have 128 filters, and the third should have 256. Also, we are adding an additional Linear layer with 512 nodes to the model architecture.

**Hint on Linear input size:** the input images are 28x28. After three MaxPool2d((2,2)) layers the spatial size becomes 28 -> 14 -> 7 -> 3 (integer division), so the flattened size going into the first Linear layer is ``3 * 3 * 256``.

### ***TODO*** Model 4

In [ ]:
def build_model_4():
  # Implement your model here. You can copy code from previous models as a reference.
  pass

### ***TODO*** Global Hyperparameters

Larger models generally require more training as there are more parameters, so you may need to increase the number of training epochs for this model. In addition, more complicated architectures and problem domains can benefit from lower learning rates. Experiment with the learning rate to see if you notice any improvements. Keep in mind that decreasing the learning rate increases the training time, and to fully train your model you may need to increase the number of epochs as well.

In [ ]:
EPOCHS = 25
LEARNING_RATE = 0.0001

### Model Compilation and Summary


In [ ]:
# get network
model_4 = build_model_4().to(device)

# print summary
print("\nModel 4 Architecture\n")
print(model_4)
print("Total parameters:", sum(p.numel() for p in model_4.parameters()))

### Testing Model 4

This model will likely take considerably longer to train (10-30 minutes depending on number of epochs, and whichever GPU Colab assigned to your runtime) . It may be a good time to take a coffee break, or work on your report for a bit.


In [ ]:
# get fresh network
model_4 = build_model_4()

perf_and_test(model_4, "Model 4")